In [ ]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList

# ==========================================
# 1. 支持 GPU 的 KGW 水印生成处理器
# ==========================================
class KGWLogitsProcessor(LogitsProcessor):
    def __init__(self, gamma=0.5, delta=2.0, hash_key=42, device='cpu'):
        self.gamma = gamma
        self.delta = delta
        self.hash_key = hash_key
        self.device = device

    def _get_greenlist_ids(self, prev_token_id, vocab_size):
        # 核心：使用指定的 device 创建 Generator，避免跨设备通信
        gen = torch.Generator(device=self.device)
        gen.manual_seed(int(self.hash_key * prev_token_id))
        #生成vocab的随机排列
        indices = torch.randperm(vocab_size, generator=gen, device=self.device)
        greenlist_size = int(self.gamma * vocab_size)
        return indices[:greenlist_size]

    def __call__(self, input_ids, scores):
        vocab_size = scores.shape[-1]
        # 处理 Batch 中的每一行
        for b in range(input_ids.shape[0]):
            prev_token = input_ids[b, -1].item()
            greenlist_ids = self._get_greenlist_ids(prev_token, vocab_size)
            # 在 GPU 上直接操作 Logits
            scores[b, greenlist_ids] += self.delta
        return scores

# ==========================================
# 2. 改进后的 KGW 检测器
# ==========================================
class KGWDetector:
    def __init__(self, tokenizer, gamma=0.5, hash_key=42, device='cpu'):
        self.tokenizer = tokenizer
        self.gamma = gamma
        self.hash_key = hash_key
        self.vocab_size = tokenizer.vocab_size
        self.device = device

    def _get_greenlist_ids(self, prev_token_id):
        gen = torch.Generator(device=self.device)
        gen.manual_seed(int(self.hash_key * prev_token_id))
        indices = torch.randperm(self.vocab_size, generator=gen, device=self.device)
        return indices[:int(self.gamma * self.vocab_size)]

    def detect(self, full_token_ids, prompt_len):
        """
        Args:
            full_token_ids: 包含 Prompt 和生成的完整 Token ID 序列 (Tensor)
            prompt_len: Prompt 部分的长度
        """
        # 只对生成部分进行循环，但需要从 prompt 的最后一个词开始算哈希
        green_token_count = 0
        total_gen_len = len(full_token_ids) - prompt_len

        if total_gen_len <= 0:
            return {"z_score": 0, "is_watermarked": False}

        for i in range(prompt_len, len(full_token_ids)):
            prev_token = full_token_ids[i-1].item()
            current_token = full_token_ids[i].item()
            greenlist = self._get_greenlist_ids(prev_token)
            
            if current_token in greenlist:
                green_token_count += 1

        # Z-score 计算
        expected = self.gamma * total_gen_len
        std = (total_gen_len * self.gamma * (1 - self.gamma)) ** 0.5
        z_score = (green_token_count - expected) / (std + 1e-6)

        return {
            "gen_len": total_gen_len,
            "green_hits": green_token_count,
            "z_score": round(float(z_score), 4),
            "is_watermarked": z_score > 4.0
        }

In [ ]:
from transformers import (
    LogitsProcessorList, 
    KGWLogitsProcessor, 
    SynthIDLogitsProcessor, 
    WatermarkingConfig
)

def get_watermark_processor(method_name, **kwargs):
    if method_name == "KGW":
        # 假设你已经定义或导入了 KGWLogitsProcessor
        return KGWLogitsProcessor(**kwargs)
    
    elif method_name == "SynthID":
        # SynthID 需要通过 WatermarkingConfig 来初始化
        # 这里的 kwargs 建议包含 keys, sampling_table_size, context_width 等
        config = WatermarkingConfig(
            name="synthid",
            keys=kwargs.get("keys", [654, 321, 123]),
            sampling_table_size=kwargs.get("sampling_table_size", 65536),
            context_width=kwargs.get("context_width", 2)
        )
        # 从配置创建处理器
        return SynthIDLogitsProcessor(config)
        
    else:
        raise ValueError(f"Unknown method: {method_name}")

In [ ]:
class WatermarkEngine:
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def generate_pair(self, prompts, processor, gen_config):
        results = []
        for prompt in tqdm(prompts, desc="Generating"):
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
            
            # 生成 Baseline
            out_base = self.model.generate(**inputs, **gen_config)
            
            # 生成 Watermarked
            out_wm = self.model.generate(
                **inputs, 
                logits_processor=LogitsProcessorList([processor]), 
                **gen_config
            )
            
            results.append({
                "prompt": prompt,
                "baseline_tokens": out_base[0].tolist(),
                "watermarked_tokens": out_wm[0].tolist(),
                "prompt_len": inputs.input_ids.shape[-1]
            })
        return results

In [ ]:
class WatermarkEvaluator:
    def __init__(self, detector, ppl_model, ppl_tokenizer):
        self.detector = detector
        self.ppl_model = ppl_model
        self.ppl_tokenizer = ppl_tokenizer

    def evaluate(self, gen_results, human_data, threshold=4.0):
        metrics = {
            "TPR": 0,          # 真正率 (准确率)
            "FPR_Base": 0,     # 误报率 (模型生成)
            "FPR_Human": 0,    # 误报率 (人类文本)
            "PPL_Base": [],
            "PPL_WM": []
        }
        
        # 1. 评估生成的数据
        for item in tqdm(gen_results, desc="Evaluating Gen Data"):
            # 检测
            z_base = self.detector.detect(torch.tensor(item["baseline_tokens"]), item["prompt_len"])["z_score"]
            z_wm = self.detector.detect(torch.tensor(item["watermarked_tokens"]), item["prompt_len"])["z_score"]
            
            if z_wm > threshold: metrics["TPR"] += 1
            if z_base > threshold: metrics["FPR_Base"] += 1
            
            # 困惑度 (需解码后计算)
            txt_base = tokenizer.decode(item["baseline_tokens"][item["prompt_len"]:])
            txt_wm = tokenizer.decode(item["watermarked_tokens"][item["prompt_len"]:])
            metrics["PPL_Base"].append(self.calc_ppl(txt_base))
            metrics["PPL_WM"].append(self.calc_ppl(txt_wm))

        # 2. 评估人类文本 (误报率测试)
        for h_txt in human_data:
            # 假设人类文本视为只有生成部分，prev_token 取一个固定起始值
            tokens = tokenizer.encode(h_txt, return_tensors="pt")[0]
            z_human = self.detector.detect(tokens, prompt_len=1)["z_score"]
            if z_human > threshold: metrics["FPR_Human"] += 1
            
        for item in tqdm(gen_results, desc="Evaluating SynthID"):
        # SynthID 检测器通常需要 input_ids
        # 注意：SynthID 检测通常返回的是 log_likelihood 差异
            res_base = self.detector.compute_log_likelihood_ratio(
            torch.tensor(item["baseline_tokens"]).unsqueeze(0)
        )
        res_wm = self.detector.compute_log_likelihood_ratio(
            torch.tensor(item["watermarked_tokens"]).unsqueeze(0)
        )
        
        score_base = res_base.item() # 获取标量分值
        score_wm = res_wm.item()
        
        if score_wm > threshold: metrics["TPR"] += 1
        if score_base > threshold: metrics["FPR_Base"] += 1
        # 汇总
        count = len(gen_results)

        return {
            "Accuracy (TPR)": metrics["TPR"] / count,
            "FPR (vs Baseline)": metrics["FPR_Base"] / count,
            "FPR (vs Human)": metrics["FPR_Human"] / len(human_data),
            "Avg PPL (Base)": np.mean(metrics["PPL_Base"]),
            "Avg PPL (WM)": np.mean(metrics["PPL_WM"])
        }

    def calc_ppl(self, text):
        # 使用评估模型计算 PPL
        inputs = self.ppl_tokenizer(text, return_tensors="pt").to(self.ppl_model.device)
        with torch.no_grad():
            loss = self.ppl_model(**inputs, labels=inputs["input_ids"]).loss
        return torch.exp(loss).item()

In [ ]:
def main():
    # 初始化组件
    # 检测设备
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"正在使用设备: {device}")
    model_id1 = "/root/autodl-tmp/Qwen2.5-1.5B-Instruct" # 
    tokenizer1 = AutoTokenizer.from_pretrained(model_id1)
    model1 = AutoModelForCausalLM.from_pretrained(model_id1).to(device)
    model_id2 = "/root/autodl-tmp/Qwen2.5-7B-Instruct" # 
    tokenizer2 = AutoTokenizer.from_pretrained(model_id2)
    model2 = AutoModelForCausalLM.from_pretrained(model_id2).to(device)
    engine = WatermarkEngine(model1, tokenizer1, device)
    detector = KGWDetector(tokenizer1, device=device)
    evaluator = WatermarkEvaluator(detector, model2, tokenizer2) # 也可以用更大的模型算 PPL
    
    # 步骤 1: 批量生成 (根据 prompts.json)
    prompts = json.load(open("prompts.json"))["test_prompts"]
    # 初始化处理器
    processor = get_watermark_processor(
    "SynthID", 
    keys=[11, 22, 33], 
    context_width=2
    )
    processor = get_watermark_processor("KGW", delta=5.0, device=device)
    raw_results = engine.generate_pair(prompts, processor, gen_config)
    
    # 步骤 2: 评估
    human_texts = json.load(open("human_texts.json"))["texts"]
    final_metrics = evaluator.evaluate(raw_results, human_texts)
    
    print(json.dumps(final_metrics, indent=4))

if __name__ == "__main__":
    main()